# A first look at decision models

*Ask a model a typed question, get a typed answer with a probability on it, in a few hundred milliseconds.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omnifroodle/couchbase_notebooks/blob/main/notebooks/experiments/01_first_look_at_decision_models.ipynb)
[![Open in Codespaces](https://img.shields.io/badge/Open%20in%20Codespaces-2f363d?logo=github&logoColor=white)](https://codespaces.new/omnifroodle/couchbase_notebooks?quickstart=1)

**Experiment.** Not a finished lesson. The model is new, in limited access, and reached through an
alpha endpoint that may change under this notebook.
**Claim.** A decision model answers narrow questions about text (yes or no, pick one, place on a
scale) with calibrated probabilities instead of prose, fast and cheap enough to sit in the request
path of a search.
**Result.** Answers in about a quarter of a second, network included, and twelve questions in one call take no longer than one. Confidence is 1.00 on clear messages and falls below 0.5 on ambiguous ones. Identical calls move the probabilities in the second decimal place.
**Requires.** decision-model
**Read** ~8 min · **Run** ~1 min · **Cost** under $0.01, about 60 calls at a few thousandths of a cent each

A chat model is a poor fit for the small judgements inside an application: *is this urgent*,
*which team handles this*, *which department is this search for*. You ask for JSON and parse
prose, the answer comes back in a second or two, and you get no sense of how sure it was.

A **decision model** is built for those judgements. You send some text (the *state*) and a set of
typed questions. It sends back typed answers, never free text, each with the probabilities
behind it. This notebook uses [TypeSafe's Jev](https://typesafe.ai/blog/introducing-system-one-models-and-jev),
the first of these, through [OpenRouter](https://openrouter.ai).

There are three kinds of question:

| type | asks | answers with |
| --- | --- | --- |
| `noul` | is this true? | the probability of yes |
| `choice` | which of these options? | the most likely option, a probability for every option, and a confidence |
| `score` | where on this scale? | a position between your levels, a probability for each level, and a confidence |

The rest of this notebook asks all three, looks at what comes back, checks how fast it is, and
ends on the use that brought us here: reading the intent behind a search query.

**Setup.** This needs an [OpenRouter API key](https://openrouter.ai/settings/keys) in
`OPENROUTER_API_KEY`, and an account with access to `typesafe/jev`. It is separate from whichever
provider the other notebooks use. No Couchbase cluster is needed.

In [ ]:
# Setup: load the helpers, install anything missing, check what this notebook needs.
import os
import pathlib
import subprocess
import sys

# Cloned on Colab, where there is no local checkout. Override to test a fork.
REPO_URL = os.environ.get("CBNB_REPO_URL", "https://github.com/omnifroodle/couchbase_notebooks")

try:
    import cbnb
except ModuleNotFoundError:
    here = pathlib.Path.cwd()
    root = next((p for p in [here, *here.parents] if (p / "cbnb" / "__init__.py").exists()), None)
    if root is None:
        # Colab: clone the repo so the committed datasets come with it.
        subprocess.check_call(["git", "clone", "--depth", "1", "--quiet", REPO_URL, "cbnb-repo"])
        root = pathlib.Path("cbnb-repo").resolve()
    sys.path.insert(0, str(root))
    import cbnb

settings = cbnb.bootstrap(requires=["decision-model"])

## 1. One call, three questions

A support message, and one question of each type about it. The question ids (`is_urgent`,
`department`, `frustration`) are for your code only; the model never sees them, so each
question's `instructions` has to stand on its own.

In [2]:
# Ask one question of each type about a support message, in a single call.
from cbnb.decisions import choice, decide, noul, score

ticket = "Help! My payouts have been failing for 3 days."
questions = {
    "is_urgent": noul("Does this message convey urgency?",
                      true="Explicitly time-sensitive", false="No urgency expressed"),
    "department": choice("Which team should handle this?", {
        "billing": "Payments, invoicing, refunds",
        "technical": "Bugs, outages, integrations",
        "sales": "Pricing, upgrades, new accounts",
    }),
    "frustration": score("How frustrated is the customer?", ["Calm", "Frustrated", "Very angry"]),
}

result = decide(ticket, questions)
for question_id in questions:
    print(f"{question_id:12} {result.value(question_id)}")
print(f"\n{result.model} · {result.seconds * 1000:.0f} ms · "
      f"{result.usage['input_tokens']} tokens in · ${result.cost:.6f}")

is_urgent    0.95
department   billing
frustration  1.04

typesafe/jev-1.13-20260917 · 254 ms · 427 tokens in · $0.000018


Three answers your code can use as they are: a probability to put a threshold on, one of the
three team names you supplied, and a number between 0 and 2. Nothing had to be parsed out of a
sentence, and the answer cannot be a team you did not list.

## 2. What came back

The headline values are the short version. The full answers carry the probabilities behind them.

In [3]:
# Show the full answers, probabilities included.
import json

print(json.dumps(result.answers, indent=2))

{
  "is_urgent": {
    "type": "noul",
    "noul": 0.95
  },
  "department": {
    "type": "choice",
    "choice": "billing",
    "probabilities": {
      "technical": 0.14,
      "billing": 0.86,
      "sales": 0
    },
    "confidence": 0.79
  },
  "frustration": {
    "type": "score",
    "score": 1.04,
    "legend": {
      "0": "Calm",
      "1": "Frustrated",
      "2": "Very angry"
    },
    "probabilities": {
      "0": 0,
      "1": 0.96,
      "2": 0.04
    },
    "confidence": 0.94
  }
}


Reading them:

- **`noul`** is the probability that the answer is *yes*. There is no separate confidence: a value
  near 0.5 *is* the model saying it does not know.
- **`choice`** is simply the option with the highest probability. The probabilities over all the
  options add up to 1.
- **`score`** is not a level. It is the probability-weighted average of the level numbers, so it
  can land between two levels. The cell below checks that.
- **`confidence`** (on `choice` and `score`) summarises how concentrated the probabilities are:
  near 1 when one option takes almost all of it, lower as it spreads out.

In [4]:
# Recompute the frustration score from its per-level probabilities.
frustration = result["frustration"]
weighted = sum(int(level) * p for level, p in frustration["probabilities"].items())
print(f"reported score {frustration['score']}, probability-weighted mean of the levels {weighted:.2f}")

reported score 1.04, probability-weighted mean of the levels 1.04


## 3. Confidence: the model saying it isn't sure

The same `department` question, over messages that range from obvious to hopeless. The
interesting column is `confidence`.

In [5]:
# Route five messages, from clear to ambiguous, and compare the confidence of each answer.
import pandas as pd

messages = [
    "I was charged twice for my subscription this month. Please refund one of them.",
    "The API returns a 500 error every time we call the webhook endpoint.",
    "We'd like to add 40 seats. What would that cost on the annual plan?",
    "My card was charged for the upgrade but it never activated, and now the dashboard shows an error.",
    "Hi, can someone call me back?",
]

rows = []
for message in messages:
    answer = decide(message, {"department": questions["department"]})["department"]
    rows.append({"message": message, "choice": answer["choice"],
                 **answer["probabilities"], "confidence": answer["confidence"]})
routing = pd.DataFrame(rows)[["message", "choice", "billing", "technical", "sales", "confidence"]]
routing.style.format(precision=2).hide(axis="index")

message,choice,billing,technical,sales,confidence
I was charged twice for my subscription this month. Please refund one of them.,billing,1.00,0.00,0.00,1.00
The API returns a 500 error every time we call the webhook endpoint.,technical,0.00,1.00,0.00,1.00
We'd like to add 40 seats. What would that cost on the annual plan?,sales,0.00,0.00,1.00,1.00
"My card was charged for the upgrade but it never activated, and now the dashboard shows an error.",technical,0.20,0.66,0.14,0.49
"Hi, can someone call me back?",technical,0.01,0.63,0.36,0.44


Every row gets an answer, because a `choice` always picks something. What changes is how the
probability is spread. The first three messages each put all of it on one team. The fourth
touches all three teams, a charge, an upgrade and an error, and the fifth says nothing about
what the caller wants. Both still get a team, and both get a confidence below 0.5.

This is the part a chat model does not give you. Your code can act on a confident answer and
send a doubtful one to a person, or to a slower model. Where to set that threshold depends on
what a wrong answer costs. That is a decision for your code, not the model.

## 4. How fast, and how steady

Twenty identical calls on one reused connection. The times include the network round trip from
wherever this notebook runs, so yours will differ. The same loop also records the `billing`
probability each time, to see whether an identical question gets an identical answer.

In [6]:
# Send the same question 20 times and record the time and the billing probability.
import statistics

import httpx

department_only = {"department": questions["department"]}
with httpx.Client() as client:
    decide(ticket, department_only, client=client)  # opens the connection; not timed
    runs = [decide(ticket, department_only, client=client) for _ in range(20)]

ms = sorted(run.seconds * 1000 for run in runs)
billing = [run["department"]["probabilities"]["billing"] for run in runs]
print(f"round trip: median {statistics.median(ms):.0f} ms, fastest {ms[0]:.0f} ms, "
      f"slowest {ms[-1]:.0f} ms")
print(f"billing probability: {min(billing)} to {max(billing)} across 20 identical calls")
print(f"calls that needed a retry: {sum(run.attempts > 1 for run in runs)}")

round trip: median 228 ms, fastest 169 ms, slowest 766 ms
billing probability: 0.86 to 0.9 across 20 identical calls
calls that needed a retry: 0


Two things to take from that.

**Latency is in the range where it can sit inside a request.** A search that consults it before
running adds a fraction of a second, not the second or more a chat model's round trip takes.
Expect the occasional slow call; the client retries a request that hangs, and the last line says
whether any did.

**Identical calls do not give identical probabilities.** They move in the second decimal place.
The decision itself held steady here, but a threshold set at exactly the value you saw once will
flicker for inputs that sit right on it. Leave some margin.

### More questions, same wait

The questions in one call are answered in parallel and independently of each other. So asking a
dozen questions should cost more tokens but take about as long as asking one.

In [7]:
# Time one question against twelve in a single call, ten calls each.
many = dict(department_only)
for topic in ["money", "a deadline", "a bug", "a refund", "an outage", "a person",
              "anger", "a product", "a number", "a length of time", "a competitor"]:
    many[f"mentions {topic}"] = noul(f"Does the message mention {topic}?")

timings = []
with httpx.Client() as client:
    decide(ticket, department_only, client=client)  # opens the connection; not timed
    for label, asked in [("1 question", department_only), (f"{len(many)} questions", many)]:
        calls = [decide(ticket, asked, client=client) for _ in range(10)]
        timings.append({"asked": label,
                        "median ms": statistics.median(c.seconds * 1000 for c in calls),
                        "input tokens": calls[0].usage["input_tokens"],
                        "cost per call ($)": calls[0].cost})
pd.DataFrame(timings).style.format({"median ms": "{:.0f}", "cost per call ($)": "{:.6f}"}).hide(axis="index")

asked,median ms,input tokens,cost per call ($)
1 question,243,349,0.000015
12 questions,207,503,0.000021


That changes how you design with it. You do not have to decide up front which questions matter.
Ask every question your code *might* need, including ones that only matter for one branch, and
let the code read the answers it uses.

## 5. Toward query intent

This is why the model is here at all. [`retrieval/02`](../retrieval/02_which_parts_helped.ipynb)
filtered searches by a department it guessed from catalogue statistics, and the guess was
sometimes wrong in ways that cost results. A decision model can read the query itself, and say
how sure it is.

Twelve real queries from the WANDS benchmark, three questions each: which department, how
specific the search is, and whether it names an attribute. The departments are the twelve largest
in the catalogue, plus `other`.

In [8]:
# Ask three intent questions about twelve real WANDS search queries.
from cbnb.datasets import load_wands_eval_set

evalset = load_wands_eval_set()
departments = evalset.products.category_hierarchy.dropna().str.split(" / ").str[0].value_counts()
departments = list(departments.index[:12])

intent = {
    "department": choice(
        "Which department of a home-goods store would a shopper searching for this be looking in?",
        {name: None for name in departments} | {"other": "None of these departments"}),
    "specificity": score("How specific is this search?", [
        "Browsing a broad category",
        "A type of product",
        "A type of product with attributes like colour, size or style",
        "One particular product or brand",
    ]),
    "names_attribute": noul("Does the search name a colour, material, size or style?"),
}

rows = []
with httpx.Client() as client:
    for query in evalset.queries["query"].sample(12, random_state=7):
        answers = decide(query, intent, client=client)
        ranked = sorted(answers["department"]["probabilities"].items(), key=lambda kv: -kv[1])
        rows.append({"query": query,
                     "department": ranked[0][0], "p": ranked[0][1],
                     "runner-up": ranked[1][0], "p ": ranked[1][1],
                     "confidence": answers["department"]["confidence"],
                     "specificity": answers.value("specificity"),
                     "names attribute": answers.value("names_attribute")})
pd.DataFrame(rows).style.format(precision=2).hide(axis="index")

query,department,p,runner-up,p,confidence,specificity,names attribute
camper,Outdoor,0.97,other,0.03,0.96,0.90,0.27
wall design shelf,Storage & Organization,0.74,Décor & Pillows,0.15,0.71,1.03,0.11
huge bookcase,Furniture,0.96,Storage & Organization,0.04,0.96,1.91,0.94
rustic storage cabinet,Storage & Organization,0.56,Furniture,0.44,0.52,1.97,0.95
gravity feeder,other,0.46,Kitchen & Tabletop,0.37,0.41,1.00,0.12
large spoon and fork wall decor,Décor & Pillows,0.85,Kitchen & Tabletop,0.15,0.84,1.24,0.19
fleur de lis living candle wall sconce bronze,Lighting,0.96,Décor & Pillows,0.04,0.95,2.07,0.97
foutains with brick look,Outdoor,0.83,Décor & Pillows,0.08,0.80,1.98,0.94
ayesha curry kitchen,Kitchen & Tabletop,0.87,other,0.13,0.86,2.69,0.11
trundle daybed,Furniture,0.86,Bed & Bath,0.14,0.84,1.04,0.39


Look at the rows with low confidence, and at the runner-up next to them. *rustic storage cabinet*
splits almost evenly between Storage & Organization and Furniture, and it really could be either.
A filter would have to bet on one, but a router that sees two close candidates can search both.
*gravity feeder* is a pet bowl, and Pet is not one of the twelve departments on offer. The model
answered `other`, the honest option, with little confidence.

The other two columns suggest different handling. *ayesha curry kitchen* names a brand and scores
highest on `specificity`: a search where a keyword match on the exact words should count for more.
The queries likely to name an attribute (*huge*, *bronze*, *rustic*) are candidates for turning
that attribute into a filter.

None of that is measured here. Whether reading intent this way beats `retrieval/02`'s
statistical guess is a question for a notebook with the relevance judgements in it.

## Where to take this

- **Query intent, measured.** Route WANDS queries by department with this model, filter or
  boost on the answer, and score it against `retrieval/02`'s guess with the same judgements.
  Low-confidence queries skip the filter.
- **A model router.** A `score` for how hard a question is, a `choice` of which model or tool
  should answer it: cheap requests go to a cheap path, and the confidence says when to escalate.
- **Query planning.** Several `noul`s in one call (*does it name a price? a brand? a size?*)
  decide which filters and which retrievers a search plan gets, in less time than the search
  itself.
- **Graph traversal.** At each node, a `choice` over the outgoing edges, with probabilities to
  explore more than one path when the model is unsure.
- **Hierarchies.** A `choice` accepts up to 255 options, but a taxonomy like WANDS's has 1,623
  categories. Choose level by level, keeping the best few paths at each level.